In [0]:
# Catalog Name
catalog = "workspace"

# cdc column
cdc_col = "modifiedDate"

surrogate_key = "booking_date"

# source object
source_object = "silver_bookings"

# source schema 
source_schema = "silver"

# target schema
target_schema = "gold"

# target object
target_object = "FactBookings"

fact_table = f"{catalog}.{source_schema}.{source_object}"

fact_key_cols = ["DimPassengersKey", "DimFlightsKey", "DimAirportsKey","booking_date"]

In [0]:
dimensions = [
 {
  "table" : f"{catalog}.{target_schema}.DimPassengers",
  "alias" : "DimPassengers",
  "join_keys" : [("passenger_id","passenger_id")]   
 },
 {
  "table" : f"{catalog}.{target_schema}.DimFlights",
  "alias" : "DimFlights",
  "join_keys" : [("flight_id","flight_id")]      
 },
 {
  "table" : f"{catalog}.{target_schema}.DimAirports",
  "alias" : "DimAirports",
  "join_keys" : [("airport_id","airport_id")]      
 }   
]
fact_columns = ["amount","booking_date","modifiedDate"]

# back-dated refresh
backdated_refresh = ""

### **Last Load Date**

In [0]:
# No back-dated refresh
if len(backdated_refresh) == 0:
    if spark.catalog.tableExists(f"workspace.{target_schema}.{target_object}"):
        last_load = spark.sql(f"SELECT max({cdc_col}) FROM workspace.{target_schema}.{target_object}").collect()[0][0]
    else:
        last_load = "1900-01-01 00:00:00"
else:
    last_load = backdated_refresh     
last_load

### **DYNAMIC FACT QUERY (BRING KEYS)**

In [0]:
def generate_fact_query_incremental(fact_table, dimensions, fact_columns, cdc_column, processing_date):

    fact_alias = "f"

    select_cols = [f"{fact_alias}.{col}" for col in fact_columns]

    join_clauses = []

    for dim in dimensions:
        table_full = dim["table"]
        alias = dim["alias"]

        table_name = table_full.split('.')[-1]
        surrogate_key = f"{alias}.{table_name}Key"

        select_cols.append(surrogate_key)

        on_conditions = [
            f"{fact_alias}.{fk} = {alias}.{dk}"
            for fk, dk in dim["join_keys"]
        ]

        join_clause = (
            f"LEFT JOIN {table_full} {alias} ON "
            + " AND ".join(on_conditions)
        )

        join_clauses.append(join_clause)

    select_clause = ",\n  ".join(select_cols)
    joins = "\n".join(join_clauses)

    where_clause = f"{fact_alias}.{cdc_column} >= DATE('{processing_date}')"

    query = f"""
    SELECT
      {select_clause}
    FROM
      {fact_table} {fact_alias}
      {joins}
    WHERE
      {where_clause}
    """.strip()

    return query

In [0]:
query = generate_fact_query_incremental(fact_table, dimensions, fact_columns, cdc_col, last_load)
query

### **DF FACT**

In [0]:
print(query)

In [0]:
df_fact = spark.sql(query)

In [0]:
df_fact.display()

In [0]:
df_fact.groupBy('DimPassengersKey','DimFlightsKey','DimAirportsKey').count().display()

### **UPSERT**

In [0]:
# Fact Key Columns Merge Condition
fact_key_cols

In [0]:
fact_key_cols_str = " AND ".join([f"src.{col} = trg.{col}" for col in fact_key_cols])
fact_key_cols_str

In [0]:
from delta.tables import DeltaTable

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col, desc, current_timestamp

# Check and remove duplicate surrogate keys
window_spec_sk = Window.partitionBy(surrogate_key) \
    .orderBy(desc(cdc_col))

df_fact = df_fact.withColumn(
    "rn",
    row_number().over(window_spec_sk)
).filter(
    col("rn") == 1
).drop("rn")

duplicate_keys = df_fact.groupBy(surrogate_key) \
    .count() \
    .filter(col("count") > 1)

if duplicate_keys.limit(1).count() > 0:
    raise ValueError("Duplicate surrogate keys found in source!")

# 3. Check if target exists
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    dlt_obj = DeltaTable.forName(spark,f"{catalog}.{target_schema}.{target_object}")
    update_set = {
        c: f"src.`{c}`"
        for c in df_fact.columns
        if c not in ["create_date", "update_date"]
    }

    update_set["update_date"] = "current_timestamp()"

    insert_set = {
        c: f"src.`{c}`"
        for c in df_fact.columns
        if c not in ["create_date", "update_date"]
    }

    insert_set["create_date"] = "current_timestamp()"
    insert_set["update_date"] = "current_timestamp()"

    # 4. Merge
    dlt_obj.alias("trg").merge(
        df_fact.alias("src"),
        fact_key_cols_str
    ) \
    .whenMatchedUpdate(
        condition=f"src.{cdc_col} > trg.{cdc_col}",
        set=update_set
    ) \
    .whenNotMatchedInsert(
        values=insert_set
    ) \
    .execute()

else:
    # 5. Create target if it doesn't exist
    df_fact.withColumn(
        "create_date", current_timestamp()
    ).withColumn(
        "update_date", current_timestamp()
    ).write.format("delta") \
        .mode("append") \
        .saveAsTable(f"{catalog}.{target_schema}.{target_object}")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col, desc

target_table = "workspace.gold.factbookings"

# 1. Read target
df_target = spark.table(target_table)

# 2. Deduplicate using fact business keys
window_spec = Window.partitionBy(
    *fact_key_cols
).orderBy(
    desc(cdc_col),
    desc("update_date")
)

df_target_dedup = df_target.withColumn(
    "rn",
    row_number().over(window_spec)
).filter(
    col("rn") == 1
).drop("rn")

# 3. Overwrite target
df_target_dedup.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table)

# 4. Verify
print("Rows after deduplication:", spark.table(target_table).count())

In [0]:
%sql
select * from workspace.gold.factbookings